# Final training — tuned settings vs. baseline (scratch ResNet, Kaggle)

Retrains the best setting from grid search, random search and Optuna, plus the original Experiment 9 setting, for the **full 100 epochs**. All models use the same seed, data and recipe, so their accuracies can be compared directly. The winner is chosen on validation accuracy, then every model is evaluated on the test set for reporting.

**Before running:**
1. **Accelerator: GPU T4 x2** and **Internet: On** (right-hand panel).
2. **Attach the tuning results:** **Add Input → Your Work →** the tuning notebook **→ Notebook Output Files**. The notebook looks for `grid_trials.csv`, `random_trials.csv` and `optuna_trials.csv`.
3. Click **Save Version → Save & Run All (Commit)**. Expect about 1 hour per model, about 4 hours in total.

If the session stops early, attach this notebook's own output as input as well and run it again. Finished models are skipped.

In [ ]:
REPO_URL = "https://github.com/TokaElsaayed/Deep-project.git"
BRANCH = "hyperparameter-tuning"
REPO_DIR = "/tmp/Deep-project"
TUNING_DIR = "/kaggle/working/hyperparameter_tuning"   # final models go to TUNING_DIR/final
EPOCHS = 100

In [ ]:
!nvidia-smi
import torch
print(torch.__version__, "CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Turn on a GPU accelerator (GPU T4 x2) in the notebook settings."

In [ ]:
!rm -rf {REPO_DIR}
!git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log -1 --oneline
!pip install -q optuna datasets

## Copy the tuning results (and any finished final models) from the attached inputs

In [ ]:
import glob, os, shutil
os.makedirs(TUNING_DIR, exist_ok=True)
sources = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/*_trials.csv", recursive=True)})
for folder in sources:
    shutil.copytree(folder, TUNING_DIR, dirs_exist_ok=True)   # includes final/ from an earlier run of this notebook
    print("Copied", folder)
assert sources, "No tuning results found. Attach the tuning notebook's output (Add Input → Your Work)."
print(sorted(os.listdir(TUNING_DIR)))

In [ ]:
# Shows which settings will be trained.
!python src/training/final_scratch_resnet.py --tuning-dir {TUNING_DIR} --epochs {EPOCHS} --dry-run

## Train and evaluate (about 1 hour per model)

In [ ]:
!python src/training/final_scratch_resnet.py --tuning-dir {TUNING_DIR} --epochs {EPOCHS}

## Results

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open(f"{TUNING_DIR}/final/final_comparison.md").read()))
display(Image(f"{TUNING_DIR}/final/final_validation_curves.png"))